In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from lightgbm import LGBMRegressor
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

# Step 1: Load the dataset
file_path = "G:/House_Hold_Energy/dataset/household_energy_dataset.csv"  # Replace with your actual dataset path
data = pd.read_csv(file_path)

# Step 2: Handle Missing 'Power' Column
if "Power" not in data.columns:
    data["Power"] = data["Current"] * data["Voltage"]

# Step 3: Preprocessing
# Drop the 'Time' column as it is not required for prediction
if "Time" in data.columns:
    data.drop("Time", axis=1, inplace=True)

# Define feature columns and target column
feature_columns = ["Current", "Voltage", "Power", "Inst_energy_t-1", "Hour_interval", "Day"]
target_column = "Inst_energy"

# Ensure all feature columns exist
missing_columns = [col for col in feature_columns if col not in data.columns]
if missing_columns:
    raise KeyError(f"Missing columns in dataset: {missing_columns}")

X = data[feature_columns]
y = data[target_column]

# Step 4: Load or Train the Model (If you have a trained model, you can skip this part)
# Train the model (if you don't have a pre-trained model)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    learning_rate=0.1,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42
)
model.fit(X_scaled, y)

# Step 5: Prepare New Data for Prediction
# Example input data for prediction
new_data = pd.DataFrame({
    "Current": [10.5],
    "Voltage": [220.0],
    "Power": [10.5 * 220.0],  # Assuming Power = Current * Voltage
    "Inst_energy_t-1": [0.015],
    "Hour_interval": [0],
    "Day": [0]
})

# Step 6: Preprocess New Data
new_data_scaled = scaler.transform(new_data)  # Scale the new data with the same scaler

# Step 7: Make Prediction
predicted_inst_energy = model.predict(new_data_scaled)

# Output the Prediction
print(f"Predicted Inst_energy: {predicted_inst_energy[0]}")

# Step 8: Apply LIME to Explain the Prediction
# Initialize the LimeTabularExplainer
explainer = LimeTabularExplainer(
    training_data=X_scaled,
    training_labels=y,
    feature_names=feature_columns,
    class_names=[target_column],  # For regression, class names can be set as target_column
    mode="regression",  # Mode is regression since we're working with a regression model
    discretize_continuous=True  # Handle continuous variables (you can turn it off if needed)
)

# Explain the prediction for the new instance
lime_exp = explainer.explain_instance(new_data_scaled[0], model.predict, num_features=6)

# Step 9: Plot LIME Explanation
lime_fig = lime_exp.as_pyplot_figure()

# Save the plot as an image file
plot_path = "lime_explanation_plot.png"  # File path to save the LIME plot
lime_fig.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close(lime_fig)

print(f"LIME explanation plot saved to {plot_path}")


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.123654 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1099
[LightGBM] [Info] Number of data points in the train set: 86400, number of used features: 6
[LightGBM] [In

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from lightgbm import LGBMRegressor
import shap
import matplotlib.pyplot as plt

# Step 1: Load the dataset
file_path = "G:/House_Hold_Energy/dataset/household_energy_dataset.csv"  # Replace with your actual dataset path
data = pd.read_csv(file_path)

# Step 2: Handle Missing 'Power' Column
if "Power" not in data.columns:
    data["Power"] = data["Current"] * data["Voltage"]

# Step 3: Preprocessing
# Drop the 'Time' column as it is not required for prediction
data.drop("Time", axis=1, inplace=True)

# Define feature columns and target column
feature_columns = ["Current", "Voltage", "Power", "Inst_energy_t-1", "Hour_interval", "Day"]
target_column = "Inst_energy"

# Ensure all feature columns exist
missing_columns = [col for col in feature_columns if col not in data.columns]
if missing_columns:
    raise KeyError(f"Missing columns in dataset: {missing_columns}")

X = data[feature_columns]
y = data[target_column]

# Step 4: Load or Train the Model (If you have a trained model, you can skip this part)
# Train the model (if you don't have a pre-trained model)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    learning_rate=0.1,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42
)

model.fit(X_scaled, y)

# Step 5: Prepare New Data for Prediction
# Example input data for prediction
new_data = pd.DataFrame({
    "Current": [10.5],
    "Voltage": [220.0],
    "Power": [10.5 * 220.0],  # Assuming Power = Current * Voltage
    "Inst_energy_t-1": [0.015],
    "Hour_interval": [0],
    "Day": [0]
})

# Step 6: Preprocess New Data
new_data_scaled = scaler.transform(new_data)  # Scale the new data with the same scaler

# Step 7: Make Prediction
predicted_inst_energy = model.predict(new_data_scaled)

# Output the Prediction
print(f"Predicted Inst_energy: {predicted_inst_energy[0]}")

# Step 8: SHAP Explanation for the new data prediction
# Create a SHAP explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the training data
shap_values = explainer.shap_values(X_scaled)

# Step 9: Generate and Save SHAP Summary Plot
# The summary plot will show the feature importance and impact of features
plt.figure()
shap.summary_plot(shap_values, X_scaled, feature_names=feature_columns, show=False)

# Save the plot as an image file
shap_plot_path = "shap_summary_plot.png"  # Specify the path for saving the SHAP plot
plt.savefig(shap_plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"SHAP summary plot saved to {shap_plot_path}")


c:\Users\tech\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007873 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1099
[LightGBM] [Info] Number of data points in the train set: 86400, number of used features: 6
[LightGBM] [In